# TinyGPT v2 — Stage 4 Continued Pretraining (Kaggle, unattended)

Stage 4 continues from the **full Stage 3 `real_latest.pt` checkpoint**: model weights plus AdamW optimizer moments and AMP GradScaler state. It starts a **new 2,000,000,000-token Stage 4 budget**, a fresh stream cursor, fresh Stage 4 counters, and a new cosine learning-rate schedule.

Attach the Stage 3 notebook output containing `/kaggle/input/notebooks/fakesiddharth/pretrain/checkpoints/real_latest.pt` and the Stage 3 `validation_cache.pt`. Enable two Tesla T4 GPUs and Internet access. This keeps the existing Stage 3 training engine: dual-GPU DataParallel, FP16 AMP, packed FineWeb-Edu streaming, atomic latest checkpoints, validation, telemetry, retry handling, and ntfy notifications.

**Data-order note:** the new shuffle seed changes the streaming order but does not mathematically guarantee zero overlap with Stage 3 documents. The deterministic MD5-based 5% validation split and pinned dataset revision are retained for comparability.

**Runtime note:** the existing 8-hour safety cutoff remains. At the current per-step batch configuration, 2B tokens needs about 61,036 optimizer steps and roughly 69,444 tokens/second to finish within 8 hours. If time expires first, `stage4_latest.pt` is the Stage 4 recovery checkpoint; publish/attach that Stage 4 output and rerun to continue the same Stage 4 stream and optimizer state.

Stage 4 saves weight-only milestone snapshots every **200M Stage 4 tokens** to keep the 5 GiB output budget safer for a 2B-token run; the full `stage4_latest.pt` checkpoint remains the primary recovery file.

In [1]:
from pathlib import Path

input_dir = Path("/kaggle/input")
STAGE3_LATEST_EXPECTED = input_dir / "notebooks/fakesiddharth/pretrain/checkpoints/real_latest.pt"

print("Datasets/notebook outputs attached to this notebook:")
if input_dir.exists():
    for item in input_dir.iterdir():
        print("📁", item)

print("\nChecking required Stage 3 continuation checkpoint...")
if STAGE3_LATEST_EXPECTED.is_file():
    print("✅ Stage 3 full checkpoint found:", STAGE3_LATEST_EXPECTED)
    print("Size:", round(STAGE3_LATEST_EXPECTED.stat().st_size / (1024**2), 1), "MiB")
else:
    print("❌ Expected checkpoint not found:", STAGE3_LATEST_EXPECTED)
    print("Attach the Stage 3 notebook output as an input using Kaggle's Add Input button.")

cache_matches = list(input_dir.rglob("validation_cache.pt")) if input_dir.exists() else []
if cache_matches:
    print("\nValidation cache candidates:")
    for path in cache_matches:
        print("✅", path, "|", round(path.stat().st_size / (1024**2), 1), "MiB")
else:
    print("\nNo attached validation_cache.pt found; the notebook will rebuild the deterministic cache if needed.")


Datasets/notebook outputs attached to this notebook:
📁 /kaggle/input/notebooks

Checking required Stage 3 continuation checkpoint...
✅ Stage 3 full checkpoint found: /kaggle/input/notebooks/fakesiddharth/pretrain/checkpoints/real_latest.pt
Size: 682.9 MiB

No attached validation_cache.pt found; the notebook will rebuild the deterministic cache if needed.


In [2]:
# Configure the environment before importing Hugging Face libraries.
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_DATASETS_DISABLE_PROGRESS_BAR"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc, hashlib, math, time, warnings, socket
from pathlib import Path
from collections import deque

import requests
import torch
from torch import nn
from torch.nn.utils import clip_grad_norm_
from transformers import AutoTokenizer, GPT2Config, GPT2LMHeadModel
from datasets import load_dataset, disable_progress_bar as disable_datasets_progress_bar
from huggingface_hub import HfApi

disable_datasets_progress_bar()

CONFIG = {
    "checkpoint_name": "real_latest.pt",
    "vocab_size": 50257, "context_length": 256, "n_embd": 512,
    "n_layer": 8, "n_head": 8, "expected_params": 51_082_752, "dropout": 0.0,
    "per_gpu_batch_size": 32, "grad_accum_steps": 2, "expected_gpu_count": 2,
    "dataset_id": "HuggingFaceFW/fineweb-edu", "dataset_config": "sample-10BT",
    "shuffle_seed": 20261024, "shuffle_buffer_size": 10000,
    "validation_fraction_percent": 5, "validation_tokens": 250000, "validation_batch_size": 16,
    "run_name": "stage4", "target_tokens": 2_000_000_000, "schedule_tokens": 2_000_000_000,
    "milestone_tokens": 200_000_000, "smoke_milestone_tokens": 5_000_000,
    "max_run_minutes": 480, "checkpoint_every_steps": 500, "log_every_steps": 50,
    "validate_every_steps": 1000, "max_warmup_steps": 150,
    "peak_lr": 2.5e-4, "floor_lr": 2.5e-5, "weight_decay": 0.01, "grad_clip_norm": 1.0,
    "output_dir": "/kaggle/working/checkpoints", "ntfy_topic": "tinygpt-siddharth-48271",
    "retry_delays_seconds": [10, 30, 60], "max_network_attempts": 10, "storage_limit_gib": 5.0,
}
assert CONFIG["schedule_tokens"] >= CONFIG["target_tokens"]
if CONFIG["run_name"] != "smoke":
    assert CONFIG["schedule_tokens"] == CONFIG["target_tokens"]

RUN_NAME = CONFIG["run_name"]
OUTPUT_DIR = Path(CONFIG["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LATEST_PATH = OUTPUT_DIR / f"{RUN_NAME}_latest.pt"
BEST_PATH = OUTPUT_DIR / f"{RUN_NAME}_best.pt"
FINAL_PATH = OUTPUT_DIR / f"{RUN_NAME}_final.pt"
TOKENS_PER_STEP = (CONFIG["per_gpu_batch_size"] * CONFIG["expected_gpu_count"]
                   * CONFIG["context_length"] * CONFIG["grad_accum_steps"])

def notify(title, message, priority="default", tags="brain"):
    try:
        # requests encodes HTTP headers as latin-1; keep the ntfy Title header ASCII-safe.
        safe_title = str(title).encode("ascii", "replace").decode("ascii")
        r = requests.post(
            f"https://ntfy.sh/{CONFIG['ntfy_topic']}",
            data=str(message).encode("utf-8"),
            headers={"Title": safe_title, "Tags": str(tags), "Priority": str(priority)},
            timeout=5,
        )
        r.raise_for_status()
        print("[ntfy] Sent:", safe_title)
    except Exception as exc:
        print(f"[ntfy warning] {title}: {type(exc).__name__}: {exc}")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable; enable a Kaggle GPU accelerator.")
GPU_COUNT = torch.cuda.device_count()
if GPU_COUNT != CONFIG["expected_gpu_count"]:
    raise RuntimeError(f"Expected {CONFIG['expected_gpu_count']} GPUs, found {GPU_COUNT}.")
DEVICE = torch.device("cuda:0")
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda, "| GPUs:", GPU_COUNT)
for i in range(GPU_COUNT):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} "
          f"({torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GiB)")
print(f"Tokens per optimizer step: {TOKENS_PER_STEP:,}")
print(f"Target: {CONFIG['target_tokens']:,}; run: {RUN_NAME}")

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

try:
    DATASET_REVISION = HfApi(token=HF_TOKEN).dataset_info(
        repo_id=CONFIG["dataset_id"], revision="main"
    ).sha
    print("Resolved dataset revision:", DATASET_REVISION)
except Exception as exc:
    DATASET_REVISION = None
    print("Warning: revision resolution failed:", type(exc).__name__, str(exc)[:200])

TOKENIZER = AutoTokenizer.from_pretrained("gpt2", token=HF_TOKEN)
TOKENIZER.pad_token = TOKENIZER.eos_token
if len(TOKENIZER) != CONFIG["vocab_size"]:
    raise RuntimeError(f"Unexpected tokenizer vocabulary size: {len(TOKENIZER)}")

cfg = GPT2Config(
    vocab_size=CONFIG["vocab_size"], n_positions=CONFIG["context_length"],
    n_ctx=CONFIG["context_length"], n_embd=CONFIG["n_embd"],
    n_layer=CONFIG["n_layer"], n_head=CONFIG["n_head"],
    resid_pdrop=CONFIG["dropout"], embd_pdrop=CONFIG["dropout"], attn_pdrop=CONFIG["dropout"],
    bos_token_id=TOKENIZER.eos_token_id, eos_token_id=TOKENIZER.eos_token_id,
    pad_token_id=TOKENIZER.eos_token_id, use_cache=False,
)
BASE_MODEL = GPT2LMHeadModel(cfg)
try:
    BASE_MODEL.set_attn_implementation("sdpa")
    print("Requested attention implementation: PyTorch SDPA (backend selection is PyTorch-dependent).")
except Exception as exc:
    print("Warning: could not explicitly select SDPA:", type(exc).__name__, str(exc)[:200])

class LossOnlyCausalLM(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, input_ids):
        out = self.model(input_ids=input_ids, labels=input_ids, use_cache=False, return_dict=True)
        return out.loss.reshape(1)

def unwrap_model(module):
    while True:
        if isinstance(module, nn.DataParallel):
            module = module.module
        elif hasattr(module, "_orig_mod"):
            module = module._orig_mod
        elif isinstance(module, LossOnlyCausalLM):
            module = module.model
        else:
            return module

STAGE3_LATEST_PATH = Path(
    "/kaggle/input/notebooks/fakesiddharth/pretrain/checkpoints/real_latest.pt"
)

def find_stage3_latest_checkpoint():
    # Use the exact user-provided input path first; don't accidentally select a different stage's checkpoint.
    if STAGE3_LATEST_PATH.is_file():
        return STAGE3_LATEST_PATH
    found = list(Path("/kaggle/input").rglob("real_latest.pt"))
    if len(found) == 1:
        print("Warning: exact path was not present; one unambiguous real_latest.pt was found:", found[0])
        return found[0]
    if not found:
        raise FileNotFoundError(
            f"Stage 3 full checkpoint not found at {STAGE3_LATEST_PATH}. "
            "Attach the Stage 3 notebook output containing real_latest.pt."
        )
    raise RuntimeError(
        "Exact Stage 3 checkpoint path is missing and multiple real_latest.pt files exist; "
        "refusing to guess: " + ", ".join(map(str, found))
    )

def extract_state_dict(obj):
    if not isinstance(obj, dict):
        raise TypeError("Source checkpoint is not a dictionary.")
    for k in ("model_state_dict", "state_dict", "model", "model_weights"):
        v = obj.get(k)
        if isinstance(v, dict) and v and all(torch.is_tensor(x) for x in v.values()):
            obj = v
            break
    if not obj or not all(torch.is_tensor(v) for v in obj.values()):
        raise RuntimeError("Could not identify model weights in the Stage 3 checkpoint.")
    out = {}
    for k, v in obj.items():
        while k.startswith("module.") or k.startswith("_orig_mod."):
            k = k.split(".", 1)[1]
        out[k] = v
    return out

STAGE3_PATH = find_stage3_latest_checkpoint()
print("Loading full Stage 3 continuation checkpoint:", STAGE3_PATH)
stage3_payload = torch.load(STAGE3_PATH, map_location="cpu", weights_only=False)
if not isinstance(stage3_payload, dict):
    raise TypeError("real_latest.pt must be a full Stage 3 checkpoint dictionary.")
if stage3_payload.get("run_name") != "real":
    raise RuntimeError(
        f"Expected Stage 3 run_name='real', got {stage3_payload.get('run_name')!r}. "
        "Refusing to initialize Stage 4 from an unexpected run."
    )
if not isinstance(stage3_payload.get("optimizer_state_dict"), dict):
    raise RuntimeError(
        "real_latest.pt does not contain optimizer_state_dict; it cannot provide the requested AdamW continuation state."
    )

source_config = stage3_payload.get("config") or {}
if source_config.get("expected_params", CONFIG["expected_params"]) != CONFIG["expected_params"]:
    raise RuntimeError("Stage 3 checkpoint parameter-count configuration does not match Stage 4.")
source_data_config = stage3_payload.get("data_config") or {}
if source_data_config.get("dataset_id", CONFIG["dataset_id"]) != CONFIG["dataset_id"] or \
   source_data_config.get("dataset_config", CONFIG["dataset_config"]) != CONFIG["dataset_config"]:
    raise RuntimeError("Stage 3 checkpoint was trained on a different dataset/config; refusing continuation.")

weights = extract_state_dict(stage3_payload)
missing, unexpected = BASE_MODEL.load_state_dict(weights, strict=False)
if missing or unexpected:
    raise RuntimeError(f"Checkpoint/model mismatch. Missing={missing[:8]}, unexpected={unexpected[:8]}")

# Preserve AdamW moments and GradScaler state for Cell 3. Deliberately DO NOT
# transfer Stage 3 step/tokens/stream cursor: Stage 4 has its own token budget
# and a different shuffle seed, so its stream begins fresh.
STAGE3_SOURCE_OPTIMIZER_STATE = stage3_payload["optimizer_state_dict"]
STAGE3_SOURCE_SCALER_STATE = stage3_payload.get("scaler_state_dict")
STAGE3_SOURCE_TOKENS_SEEN = int(stage3_payload.get("tokens_seen", 0))
STAGE3_SOURCE_STEP = int(stage3_payload.get("step", 0))
STAGE3_DATASET_REVISION = source_data_config.get("dataset_revision")
if STAGE3_DATASET_REVISION:
    DATASET_REVISION = STAGE3_DATASET_REVISION
    print("Pinned dataset revision inherited from Stage 3 checkpoint:", DATASET_REVISION)
else:
    warnings.warn("Stage 3 checkpoint has no dataset revision; using the revision resolved by the API.")

print(f"Stage 3 source: step={STAGE3_SOURCE_STEP:,}; tokens_seen={STAGE3_SOURCE_TOKENS_SEEN:,}")
print("Stage 4 will reset stream cursor and token counters, change shuffle seed to", CONFIG["shuffle_seed"])
del stage3_payload, weights
# The optimizer/scaler sub-states remain available for restoration after optimizer construction.
gc.collect()

params = sum(p.numel() for p in BASE_MODEL.parameters())
print(f"Model parameters: {params:,}")
if params != CONFIG["expected_params"]:
    raise RuntimeError(f"Expected {CONFIG['expected_params']:,} parameters, got {params:,}.")
BASE_MODEL.to(DEVICE)
PARALLEL_MODEL = nn.DataParallel(
    LossOnlyCausalLM(BASE_MODEL), device_ids=list(range(GPU_COUNT)), output_device=0
)
print("Stage 3 model weights loaded; two-GPU DataParallel is ready for Stage 4.")


PyTorch: 2.11.0+cu128 | CUDA: 12.8 | GPUs: 2
GPU 0: Tesla T4 (14.56 GiB)
GPU 1: Tesla T4 (14.56 GiB)
Tokens per optimizer step: 32,768
Target: 2,000,000,000; run: stage4
Resolved dataset revision: 87f09149ef4734204d70ed1d046ddc9ca3f2b8f9
Requested attention implementation: PyTorch SDPA (backend selection is PyTorch-dependent).
Loading full Stage 3 continuation checkpoint: /kaggle/input/notebooks/fakesiddharth/pretrain/checkpoints/real_latest.pt
Pinned dataset revision inherited from Stage 3 checkpoint: 87f09149ef4734204d70ed1d046ddc9ca3f2b8f9
Stage 3 source: step=28,992; tokens_seen=950,009,856
Stage 4 will reset stream cursor and token counters, change shuffle seed to 20261024
Model parameters: 51,082,752
Stage 3 model weights loaded; two-GPU DataParallel is ready for Stage 4.


In [3]:
# ----------------------------- TRAINING ENGINE -------------------------------
import copy
import shutil
import urllib3
from requests.exceptions import RequestException
from urllib3.exceptions import HTTPError as Urllib3HTTPError

DATASET_ID = CONFIG["dataset_id"]
DATASET_CONFIG = CONFIG["dataset_config"]
CONTEXT = CONFIG["context_length"]
EOS_ID = TOKENIZER.eos_token_id
VALIDATION_CACHE_PATH = OUTPUT_DIR / "validation_cache.pt"
VALIDATION_CACHE_NAME = "validation_cache.pt"

def load_pt(path):
    return torch.load(path, map_location="cpu", weights_only=False)

def find_resume_checkpoint():
    candidates = [LATEST_PATH, Path("/kaggle/working") / f"{RUN_NAME}_latest.pt"]
    if Path("/kaggle/input").exists():
        candidates.extend(Path("/kaggle/input").rglob(f"{RUN_NAME}_latest.pt"))
    found, seen = [], set()
    for p in candidates:
        if p.exists() and str(p.resolve()) not in seen:
            found.append(p)
            seen.add(str(p.resolve()))
    if len(found) > 1:
        preferred = [p for p in found if p.resolve() == LATEST_PATH.resolve()]
        if len(preferred) == 1:
            return preferred[0]
        raise RuntimeError("Multiple same-run resume checkpoints found; refusing to guess: " +
                           ", ".join(map(str, found)))
    return found[0] if found else None

def stream_dataset(revision, shuffled=True):
    kwargs = dict(path=DATASET_ID, name=DATASET_CONFIG, split="train",
                  streaming=True, token=HF_TOKEN)
    if revision:
        kwargs["revision"] = revision
    ds = load_dataset(**kwargs)
    if shuffled:
        ds = ds.shuffle(seed=CONFIG["shuffle_seed"],
                        buffer_size=CONFIG["shuffle_buffer_size"])
    return ds

def row_text(row):
    x = row.get("text", "")
    return "" if x is None else (x if isinstance(x, str) else str(x))

def is_validation_doc(text):
    digest = hashlib.md5(text.encode("utf-8", errors="replace")).hexdigest()
    return int(digest[:8], 16) % 100 < CONFIG["validation_fraction_percent"]

def build_validation_cache(revision):
    print(f"Building deterministic validation cache ({CONFIG['validation_tokens']:,} target tokens)...")
    ds = stream_dataset(revision, shuffled=False)
    chunks, buf, scanned = [], [], 0
    collected = 0
    for row in ds:
        scanned += 1
        text = row_text(row)
        if not text or not is_validation_doc(text):
            continue
        ids = TOKENIZER.encode(text, add_special_tokens=False)
        if ids:
            buf.extend(ids)
        buf.append(EOS_ID)
        while len(buf) >= CONTEXT:
            chunks.append(buf[:CONTEXT])
            del buf[:CONTEXT]
            collected += CONTEXT
            if collected >= CONFIG["validation_tokens"]:
                break
        if collected >= CONFIG["validation_tokens"]:
            break
        if scanned % 100000 == 0:
            print(f"Validation scan: {scanned:,} rows; {collected:,} token positions.")
    if not chunks:
        raise RuntimeError("No validation chunks could be built from FineWeb-Edu.")
    ids_tensor = torch.tensor(chunks, dtype=torch.int32)
    metadata = {
        "dataset_id": DATASET_ID, "dataset_config": DATASET_CONFIG,
        "dataset_revision": revision, "validation_fraction_percent": CONFIG["validation_fraction_percent"],
        "context_length": CONTEXT, "validation_tokens_actual": int(ids_tensor.numel()),
        "split_method": "MD5(text) first 8 hex digits mod 100 < 5",
    }
    temp = VALIDATION_CACHE_PATH.with_name(VALIDATION_CACHE_PATH.name + ".tmp")
    torch.save({"input_ids": ids_tensor, "metadata": metadata}, temp)
    os.replace(temp, VALIDATION_CACHE_PATH)
    print(f"Saved validation cache: {VALIDATION_CACHE_PATH} ({ids_tensor.numel():,} positions)")
    return ids_tensor.long(), metadata

def get_validation_cache(revision):
    # Prefer the exact Stage 3 cache alongside the checkpoint, then search other attached inputs.
    candidates = [VALIDATION_CACHE_PATH]
    stage3_cache = Path("/kaggle/input/notebooks/fakesiddharth/pretrain/checkpoints/validation_cache.pt")
    if stage3_cache not in candidates:
        candidates.append(stage3_cache)
    if Path("/kaggle/input").exists():
        for cache_path in Path("/kaggle/input").rglob(VALIDATION_CACHE_NAME):
            if cache_path not in candidates:
                candidates.append(cache_path)
    for p in candidates:
        if not p.exists():
            continue
        try:
            obj = load_pt(p)
            if torch.is_tensor(obj):
                ids, meta = obj.long(), None
            elif isinstance(obj, dict) and "input_ids" in obj:
                ids, meta = obj["input_ids"].long(), obj.get("metadata")
            else:
                continue
            if ids.ndim != 2 or ids.shape[1] != CONTEXT:
                print("Ignoring incompatible validation cache:", p)
                continue
            if meta:
                if meta.get("dataset_id") != DATASET_ID or meta.get("dataset_config") != DATASET_CONFIG:
                    print("Ignoring cache for a different dataset:", p)
                    continue
                if meta.get("dataset_revision") and revision and meta["dataset_revision"] != revision:
                    print("Ignoring cache for a different dataset revision:", p)
                    continue
                if meta.get("validation_fraction_percent", CONFIG["validation_fraction_percent"]) != CONFIG["validation_fraction_percent"]:
                    print("Ignoring cache with a different validation fraction:", p)
                    continue
                if meta.get("context_length", CONTEXT) != CONTEXT:
                    print("Ignoring cache with a different context length:", p)
                    continue
                expected_split = "MD5(text) first 8 hex digits mod 100 < 5"
                if meta.get("split_method") and meta.get("split_method") != expected_split:
                    print("Ignoring cache with a different split method:", p)
                    continue
            else:
                warnings.warn(f"Legacy validation cache has no metadata; identity cannot be verified: {p}")
            print(f"Using validation cache {p}: {ids.numel():,} token positions.")
            return ids, meta
        except Exception as exc:
            print(f"Could not load validation cache {p}: {type(exc).__name__}: {exc}")
    return build_validation_cache(revision)

def evaluate(model, validation_ids):
    model.eval()
    losses = []
    with torch.no_grad():
        for start in range(0, validation_ids.shape[0], CONFIG["validation_batch_size"]):
            x = validation_ids[start:start + CONFIG["validation_batch_size"]].long().to(DEVICE)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                out = model(input_ids=x, labels=x, use_cache=False, return_dict=True)
            if torch.isfinite(out.loss):
                losses.append(float(out.loss.detach().cpu()))
    model.train()
    if not losses:
        raise RuntimeError("Validation did not produce finite losses.")
    return sum(losses) / len(losses)

def atomic_save(obj, path):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def save_weights(model, path):
    state = {k: v.detach().cpu().float().contiguous()
             for k, v in unwrap_model(model).state_dict().items()}
    atomic_save(state, path)
    del state
    gc.collect()

def fmt_duration(seconds):
    if not math.isfinite(seconds):
        return "estimating"
    seconds = max(0, int(seconds))
    h, rem = divmod(seconds, 3600)
    m, _ = divmod(rem, 60)
    return f"{h}h {m:02d}m"

def gpu_memory():
    return " | ".join(
        f"GPU{i} {torch.cuda.memory_allocated(i)/1024**3:.2f} GiB allocated, "
        f"{torch.cuda.max_memory_reserved(i)/1024**3:.2f} GiB peak reserved"
        for i in range(torch.cuda.device_count())
    )

def is_network_error(exc):
    if isinstance(exc, (RequestException, Urllib3HTTPError, TimeoutError,
                        ConnectionError, socket.timeout)):
        return True
    s = f"{type(exc).__name__}: {exc}".lower()
    return any(x in s for x in (
        "connection", "timed out", "timeout", "chunkedencoding", "incomplete read",
        "temporarily unavailable", "remote end closed", "http error", "502", "503",
        "504", "network", "ssl", "name resolution", "dns"
    ))

class PackedStream:
    """State is raw shuffled rows consumed plus residual packed token IDs."""
    def __init__(self, revision, docs=0, buffer=None):
        self.revision = revision
        self.docs = int(docs)
        self.buffer = list(buffer or [])
        self._open()

    def _open(self):
        attempt = 0
        while True:
            try:
                ds = stream_dataset(self.revision, shuffled=True)
                if self.docs:
                    print(f"Rebuilding stream and skipping {self.docs:,} shuffled rows...")
                    ds = ds.skip(self.docs)
                self.iterator = iter(ds)
                return
            except Exception as exc:
                if not is_network_error(exc) or attempt >= CONFIG["max_network_attempts"] - 1:
                    raise
                delay = CONFIG["retry_delays_seconds"][min(attempt, len(CONFIG["retry_delays_seconds"]) - 1)]
                attempt += 1
                print(f"[stream reopen retry {attempt}/{CONFIG['max_network_attempts']}] "
                      f"{type(exc).__name__}: {exc}; waiting {delay}s.")
                time.sleep(delay)

    def state(self):
        return {"docs_consumed": self.docs, "token_buffer": list(self.buffer)}

    def restore(self, state):
        self.docs = int(state["docs_consumed"])
        self.buffer = list(state["token_buffer"])
        self._open()

    def next_chunk(self):
        while len(self.buffer) < CONTEXT:
            row = next(self.iterator)
            self.docs += 1  # Count raw rows before validation filtering.
            text = row_text(row)
            if not text or is_validation_doc(text):
                continue
            ids = TOKENIZER.encode(text, add_special_tokens=False)
            if ids:
                self.buffer.extend(ids)
            self.buffer.append(EOS_ID)
        chunk = self.buffer[:CONTEXT]
        del self.buffer[:CONTEXT]
        return chunk

    def next_batch(self, size):
        return torch.tensor([self.next_chunk() for _ in range(size)], dtype=torch.long)

def find_resume():
    # Prefer this session's most recent local Stage 4 checkpoint. This also
    # prevents an attached older Stage 4 output from shadowing newer local work.
    if LATEST_PATH.is_file():
        return LATEST_PATH
    working_candidate = Path("/kaggle/working") / f"{RUN_NAME}_latest.pt"
    if working_candidate.is_file():
        return working_candidate
    found, seen = [], set()
    if Path("/kaggle/input").exists():
        for p in Path("/kaggle/input").rglob(f"{RUN_NAME}_latest.pt"):
            key = str(p.resolve())
            if p.is_file() and key not in seen:
                found.append(p)
                seen.add(key)
    if len(found) > 1:
        raise RuntimeError(
            "Multiple attached Stage 4 latest checkpoints found; detach older/duplicate Stage 4 outputs "
            "or attach only the intended latest checkpoint: " + ", ".join(map(str, found))
        )
    return found[0] if found else None

def make_checkpoint(step, tokens, cursor, revision, optimizer, scaler,
                    baseline, latest_val, best_val, first_hash, next_milestone):
    return {
        "format_version": 3, "run_name": RUN_NAME,
        "model_state_dict": {k: v.detach().cpu().contiguous()
                             for k, v in unwrap_model(PARALLEL_MODEL).state_dict().items()},
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "step": int(step), "tokens_seen": int(tokens),
        "docs_consumed": int(cursor["docs_consumed"]),
        "token_buffer": list(cursor["token_buffer"]),
        "config": dict(CONFIG),
        "data_config": {
            "dataset_id": DATASET_ID, "dataset_config": DATASET_CONFIG,
            "dataset_revision": revision, "shuffle_seed": CONFIG["shuffle_seed"],
            "shuffle_buffer_size": CONFIG["shuffle_buffer_size"],
            "validation_fraction_percent": CONFIG["validation_fraction_percent"],
        },
        "baseline_val_loss": baseline, "latest_val_loss": latest_val,
        "best_val_loss": best_val, "first_chunk_hash": first_hash,
        "next_milestone": int(next_milestone), "saved_at_unix": time.time(),
    }

resume_path = find_resume()
resume = load_pt(resume_path) if resume_path else None
if resume is not None:
    if not isinstance(resume, dict):
        raise RuntimeError("Resume checkpoint is not a dictionary.")
    saved_run_name = resume.get("run_name")
    if saved_run_name is None and RUN_NAME == "smoke":
        warnings.warn("Legacy smoke checkpoint has no run_name; accepting it only because the exact smoke_latest.pt filename matched.")
    elif saved_run_name != RUN_NAME:
        raise RuntimeError(f"Resume checkpoint run_name mismatch: expected {RUN_NAME!r}, found {saved_run_name!r}.")
    print("Resuming:", resume_path)
    resume_weights = resume.get("model_state_dict", resume.get("state_dict", resume.get("model")))
    if not isinstance(resume_weights, dict) or not resume_weights:
        raise RuntimeError("Resume checkpoint does not contain a recognizable model state_dict.")
    BASE_MODEL.load_state_dict(resume_weights, strict=True)
    # When resuming from an attached notebook output, bring its weight-only best
    # checkpoint into /kaggle/working so the final output preserves it too.
    if not BEST_PATH.exists():
        attached_best = Path(resume_path).with_name(f"{RUN_NAME}_best.pt")
        if attached_best.is_file():
            temp_best = BEST_PATH.with_name(BEST_PATH.name + ".tmp")
            shutil.copy2(attached_best, temp_best)
            os.replace(temp_best, BEST_PATH)
            print("Recovered prior best weights from attached Stage 4 output:", attached_best)
        else:
            warnings.warn(
                "Attached Stage 4 latest checkpoint has no sibling stage4_best.pt. "
                "The best weights from the prior session may be unavailable; preserving the latest model as a fallback."
            )
            save_weights(BASE_MODEL, BEST_PATH)
            if resume.get("latest_val_loss") is not None:
                resume["best_val_loss"] = resume["latest_val_loss"]
    saved_data = resume.get("data_config") or {}
    for key, expected in (
        ("dataset_id", DATASET_ID),
        ("dataset_config", DATASET_CONFIG),
        ("shuffle_seed", CONFIG["shuffle_seed"]),
        ("shuffle_buffer_size", CONFIG["shuffle_buffer_size"]),
        ("validation_fraction_percent", CONFIG["validation_fraction_percent"]),
    ):
        saved_value = saved_data.get(key)
        if saved_value is not None and saved_value != expected:
            raise RuntimeError(
                f"Stage 4 resume checkpoint data mismatch for {key}: saved={saved_value!r}, expected={expected!r}. "
                "Do not change the Stage 4 data configuration when resuming."
            )
    dataset_revision = saved_data.get("dataset_revision") or DATASET_REVISION
    if STAGE3_DATASET_REVISION and dataset_revision != STAGE3_DATASET_REVISION:
        raise RuntimeError("Stage 4 resume uses a different FineWeb-Edu revision than the Stage 3 continuation source.")
    if not saved_data.get("dataset_revision"):
        warnings.warn("Checkpoint lacks pinned dataset revision; exact stream identity is not guaranteed.")
    step = int(resume.get("step", 0))
    tokens_seen = int(resume.get("tokens_seen", step * TOKENS_PER_STEP))
    docs_consumed = int(resume.get("docs_consumed", 0))
    token_buffer = list(resume.get("token_buffer", []))
    baseline_val_loss = resume.get("baseline_val_loss")
    latest_val_loss = resume.get("latest_val_loss")
    best_val_loss = resume.get("best_val_loss")
    first_chunk_hash = resume.get("first_chunk_hash")
else:
    print("No Stage 4 resume checkpoint found; initializing a NEW Stage 4 run from Stage 3 real_latest.pt.")
    dataset_revision = STAGE3_DATASET_REVISION or DATASET_REVISION
    # New Stage 4 budget and a new shuffled stream: do not import Stage 3 counters/cursor.
    step = tokens_seen = docs_consumed = 0
    token_buffer = []
    baseline_val_loss = latest_val_loss = best_val_loss = first_chunk_hash = None

if not dataset_revision:
    raise RuntimeError("Could not pin FineWeb-Edu revision. Check Internet access and rerun.")

VALIDATION_IDS, VALIDATION_META = get_validation_cache(dataset_revision)
BASE_MODEL.to(DEVICE)
optimizer = torch.optim.AdamW(
    BASE_MODEL.parameters(), lr=CONFIG["peak_lr"], weight_decay=CONFIG["weight_decay"],
    betas=(0.9, 0.95), eps=1e-8
)
try:
    scaler = torch.amp.GradScaler("cuda", enabled=True)
except Exception:
    scaler = torch.cuda.amp.GradScaler(enabled=True)

if resume is not None:
    if resume.get("optimizer_state_dict") is not None:
        optimizer.load_state_dict(resume["optimizer_state_dict"])
    else:
        raise RuntimeError("Stage 4 latest checkpoint has no optimizer state; safe continuation is not possible.")
    if resume.get("scaler_state_dict"):
        try:
            scaler.load_state_dict(resume["scaler_state_dict"])
        except Exception as exc:
            warnings.warn(f"Could not restore Stage 4 GradScaler state: {exc}")
    if "docs_consumed" not in resume or "token_buffer" not in resume:
        raise RuntimeError("Stage 4 checkpoint lacks the stream cursor required for safe continuation.")
    if not first_chunk_hash:
        warnings.warn("Stage 4 checkpoint lacks first_chunk_hash; resume data self-check cannot be performed.")
    if baseline_val_loss is None:
        baseline_val_loss = evaluate(BASE_MODEL, VALIDATION_IDS)
    print("Restored Stage 4 model, AdamW moments, AMP scaler, token counters, and Stage 4 stream cursor.")
else:
    if STAGE3_SOURCE_OPTIMIZER_STATE is None:
        raise RuntimeError("Stage 3 source optimizer state is missing; cannot perform optimizer-state continuation.")
    optimizer.load_state_dict(STAGE3_SOURCE_OPTIMIZER_STATE)
    # The old optimizer moments continue, but the Stage 4 schedule starts fresh.
    # lr_for() overwrites this value before the first optimizer update.
    for group in optimizer.param_groups:
        group["lr"] = CONFIG["peak_lr"]
    if STAGE3_SOURCE_SCALER_STATE:
        try:
            scaler.load_state_dict(STAGE3_SOURCE_SCALER_STATE)
        except Exception as exc:
            warnings.warn(f"Could not restore Stage 3 GradScaler state; using fresh scaler: {exc}")
    print("Restored Stage 3 AdamW moments and AMP scaler; Stage 4 schedule/counters/stream are fresh.")
    print("Evaluating Stage 3 starting weights on the fixed validation cache...")
    baseline_val_loss = evaluate(BASE_MODEL, VALIDATION_IDS)
    latest_val_loss = baseline_val_loss
    best_val_loss = baseline_val_loss
    save_weights(BASE_MODEL, BEST_PATH)
    print(f"Stage 4 starting baseline validation loss: {baseline_val_loss:.4f}")

# Free the transferred checkpoint state once its tensors have been loaded into optimizer/scaler.
STAGE3_SOURCE_OPTIMIZER_STATE = None
STAGE3_SOURCE_SCALER_STATE = None
gc.collect()

packer = PackedStream(dataset_revision, docs_consumed, token_buffer)
committed_cursor = packer.state()
resume_check_pending = bool(resume is not None and first_chunk_hash)
first_chunk_hash_candidate = None

milestone_interval = CONFIG["milestone_tokens"] if RUN_NAME != "smoke" else CONFIG["smoke_milestone_tokens"]
next_milestone = int(resume.get("next_milestone", ((tokens_seen // milestone_interval) + 1) * milestone_interval)) if resume else milestone_interval
while next_milestone <= tokens_seen:
    next_milestone += milestone_interval

total_steps = max(1, math.ceil(CONFIG["schedule_tokens"] / TOKENS_PER_STEP))
warmup_steps = max(1, min(CONFIG["max_warmup_steps"], int(0.05 * total_steps)))
print(f"Warmup: {warmup_steps} steps | scheduled steps: {total_steps:,}")
print(f"Required average speed for target within 8 hours: "
      f"{CONFIG['target_tokens']/(CONFIG['max_run_minutes']*60):,.0f} tok/s, before overhead.")

def lr_for(tokens_before, step_number):
    if step_number <= warmup_steps:
        return CONFIG["peak_lr"] * step_number / warmup_steps
    progress = tokens_before / max(1, CONFIG["schedule_tokens"])
    if progress <= 0.85:
        return CONFIG["peak_lr"]
    p = min(1.0, (progress - 0.85) / 0.15)
    return CONFIG["floor_lr"] + (CONFIG["peak_lr"] - CONFIG["floor_lr"]) * 0.5 * (1 + math.cos(math.pi * p))

def save_latest(reason):
    payload = make_checkpoint(step, tokens_seen, committed_cursor, dataset_revision,
                              optimizer, scaler, baseline_val_loss, latest_val_loss,
                              best_val_loss, first_chunk_hash, next_milestone)
    tmp = LATEST_PATH.with_name(LATEST_PATH.name + ".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, LATEST_PATH)
    del payload
    gc.collect()
    print(f"[checkpoint] {LATEST_PATH.name}: step {step:,}, tokens {tokens_seen:,} ({reason})")

def save_milestones():
    global next_milestone
    while tokens_seen >= next_milestone:
        label = f"{next_milestone // 1_000_000}M"
        out = OUTPUT_DIR / f"{RUN_NAME}_milestone_{label}.pt"
        save_weights(BASE_MODEL, out)
        print("[milestone] Saved", out.name)
        next_milestone += milestone_interval

notify("TinyGPT Stage 4 pretraining started",
       f"Run: {RUN_NAME}\nStep: {step:,}\nTokens: {tokens_seen:,}/{CONFIG['target_tokens']:,}\n"
       f"Revision: {dataset_revision}\nGPUs: {GPU_COUNT}\nTokens/step: {TOKENS_PER_STEP:,}",
       priority="high", tags="rocket")

start_time = time.time()
last_hourly = start_time
last_speed_time = start_time
rolling_rates = deque(maxlen=20)
post_warm_time = post_warm_tokens = None
last_loss = float("nan")
stop_reason = None
network_attempts_total = 0
overflow_retries = 0
torch.cuda.reset_peak_memory_stats()
PARALLEL_MODEL.train()

try:
    while tokens_seen < CONFIG["target_tokens"]:
        if time.time() - start_time >= CONFIG["max_run_minutes"] * 60:
            stop_reason = "time_limit"
            break

        # Fetch/cache all CPU microbatches first. Stream rollback is used ONLY
        # when packer.next_batch() itself fails; AMP retries reuse these cached tensors.
        cursor_before = {
            "docs_consumed": committed_cursor["docs_consumed"],
            "token_buffer": list(committed_cursor["token_buffer"]),
        }
        step_batches = []
        attempt = 0
        while len(step_batches) < CONFIG["grad_accum_steps"]:
            try:
                cpu_batch = packer.next_batch(CONFIG["per_gpu_batch_size"] * GPU_COUNT)
                step_batches.append(cpu_batch)
            except Exception as exc:
                # Only a genuine stream/network failure triggers stream rollback.
                if not is_network_error(exc):
                    raise
                packer.restore(cursor_before)
                step_batches.clear()
                attempt += 1
                network_attempts_total += 1
                if attempt >= CONFIG["max_network_attempts"]:
                    save_latest("network retry exhaustion")
                    notify(
                        "TinyGPT urgent: stream failed",
                        f"Run: {RUN_NAME}\nError: {type(exc).__name__}: {exc}\n"
                        f"Last saved step: {step:,}\nTokens: {tokens_seen:,}",
                        priority="urgent",
                        tags="warning,rotating_light",
                    )
                    raise
                delay = CONFIG["retry_delays_seconds"][
                    min(attempt - 1, len(CONFIG["retry_delays_seconds"]) - 1)
                ]
                print(
                    f"[network retry {attempt}/{CONFIG['max_network_attempts']}] "
                    f"{type(exc).__name__}: {exc}; rolling back stream to step {step:,} "
                    f"and waiting {delay}s."
                )
                time.sleep(delay)

        step_finished = False
        while not step_finished:
            optimizer.zero_grad(set_to_none=True)
            micro_losses = []
            first_batch_hash = hashlib.sha256(step_batches[0][0].numpy().tobytes()).hexdigest()

            if resume_check_pending:
                if first_batch_hash == first_chunk_hash:
                    raise RuntimeError(
                        "Resume self-check failed: resumed first chunk equals original first chunk."
                    )
                print("Resume self-check passed: first resumed chunk differs from the original.")
                resume_check_pending = False

            # On AMP overflow, replay this same in-memory CPU batch list; do not
            # reopen or skip through the remote Hugging Face stream.
            for cpu_batch in step_batches:
                batch = cpu_batch.to(DEVICE, non_blocking=True)
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    replica_losses = PARALLEL_MODEL(batch)
                    loss = replica_losses.mean()
                if not torch.isfinite(loss):
                    raise FloatingPointError(f"Non-finite loss at step {step + 1}: {loss.item()}")
                scaler.scale(loss / CONFIG["grad_accum_steps"]).backward()
                micro_losses.append(float(loss.detach().cpu()))

            scaler.unscale_(optimizer)
            grad_norm = clip_grad_norm_(BASE_MODEL.parameters(), CONFIG["grad_clip_norm"])
            if not torch.isfinite(torch.as_tensor(grad_norm)):
                optimizer.zero_grad(set_to_none=True)
                overflow_retries += 1
                if overflow_retries > 5:
                    raise FloatingPointError("Non-finite gradient norm persisted after 5 retries.")
                try:
                    scaler.update(new_scale=max(1.0, scaler.get_scale() / 2.0))
                except Exception:
                    pass
                print(
                    f"[warning] Non-finite gradients; replaying cached batches "
                    f"({overflow_retries}/5), without reopening the data stream."
                )
                continue

            lr = lr_for(tokens_seen, step + 1)
            for group in optimizer.param_groups:
                group["lr"] = lr
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() < old_scale:
                optimizer.zero_grad(set_to_none=True)
                overflow_retries += 1
                if overflow_retries > 5:
                    raise FloatingPointError("FP16 overflow persisted after 5 retries.")
                print(
                    f"[warning] GradScaler skipped optimizer update; replaying cached batches "
                    f"({overflow_retries}/5), without reopening the data stream."
                )
                continue

            overflow_retries = 0
            step += 1
            tokens_seen += TOKENS_PER_STEP
            last_loss = sum(micro_losses) / max(1, len(micro_losses))
            committed_cursor = packer.state()
            if first_chunk_hash is None and tokens_seen == TOKENS_PER_STEP:
                first_chunk_hash = first_batch_hash
            step_finished = True

        now = time.time()
        rolling_rates.append(TOKENS_PER_STEP / max(1e-6, now - last_speed_time))
        last_speed_time = now
        if post_warm_time is None and step >= 30:
            post_warm_time, post_warm_tokens = now, tokens_seen

        save_milestones()
        if step % CONFIG["checkpoint_every_steps"] == 0:
            save_latest("periodic")

        if step % CONFIG["validate_every_steps"] == 0:
            latest_val_loss = evaluate(BASE_MODEL, VALIDATION_IDS)
            print(f"[validation] step {step:,} | tokens {tokens_seen:,} | val_loss {latest_val_loss:.4f}")
            notify("TinyGPT validation",
                   f"Run: {RUN_NAME}\nStep: {step:,}\nTokens: {tokens_seen:,}/{CONFIG['target_tokens']:,}\n"
                   f"Validation loss: {latest_val_loss:.4f}\nTraining loss: {last_loss:.4f}",
                   priority="high", tags="white_check_mark")
            if best_val_loss is None or latest_val_loss < best_val_loss:
                best_val_loss = latest_val_loss
                save_weights(BASE_MODEL, BEST_PATH)
                print(f"[best] Saved {BEST_PATH.name}; validation loss {best_val_loss:.4f}")
            save_latest("post-validation")

        if step % CONFIG["log_every_steps"] == 0:
            elapsed = now - start_time
            rolling = sum(rolling_rates) / len(rolling_rates) if rolling_rates else 0.0
            overall = ((tokens_seen - post_warm_tokens) / max(1e-6, now - post_warm_time)
                       if post_warm_time is not None else rolling)
            speed = overall if overall > 0 else rolling
            remaining = max(0, CONFIG["target_tokens"] - tokens_seen)
            eta = remaining / speed if speed > 0 else float("inf")
            pct = min(100.0, 100 * tokens_seen / CONFIG["target_tokens"])
            print(f"[train] step {step:,} | {tokens_seen:,}/{CONFIG['target_tokens']:,} ({pct:.2f}%) "
                  f"| loss {last_loss:.4f} | lr {lr:.7g} | rolling {rolling:,.0f} tok/s "
                  f"| post-warmup {overall:,.0f} tok/s | ETA {fmt_duration(eta)} "
                  f"| elapsed {fmt_duration(elapsed)} | {gpu_memory()}")
        if now - last_hourly >= 3600:
            notify("TinyGPT hourly update",
                   f"Run: {RUN_NAME}\nStep: {step:,}\nTokens: {tokens_seen:,}/{CONFIG['target_tokens']:,}\n"
                   f"Loss: {last_loss:.4f}\nElapsed: {fmt_duration(now-start_time)}",
                   priority="default", tags="clock")
            last_hourly = now

        if tokens_seen >= CONFIG["target_tokens"]:
            stop_reason = "target_reached"
            break
        if time.time() - start_time >= CONFIG["max_run_minutes"] * 60:
            stop_reason = "time_limit"
            break

    if stop_reason is None:
        stop_reason = "target_reached" if tokens_seen >= CONFIG["target_tokens"] else "stopped"

    # Final validation ensures the reported loss and best checkpoint include the
    # final weights even when the run exits between regular validation intervals.
    if step > 0 and step % CONFIG["validate_every_steps"] != 0:
        latest_val_loss = evaluate(BASE_MODEL, VALIDATION_IDS)
        print(
            f"[final validation] step {step:,} | tokens {tokens_seen:,} "
            f"| val_loss {latest_val_loss:.4f}"
        )
        if best_val_loss is None or latest_val_loss < best_val_loss:
            best_val_loss = latest_val_loss
            save_weights(BASE_MODEL, BEST_PATH)
            print(f"[best] Final validation improved; saved {BEST_PATH.name} "
                  f"with val_loss {best_val_loss:.4f}")

    save_latest(stop_reason)
    save_weights(BASE_MODEL, FINAL_PATH)
    elapsed = time.time() - start_time
    session_avg_speed = tokens_seen / max(1e-6, elapsed)
    avg_speed = (
        (tokens_seen - post_warm_tokens) / max(1e-6, time.time() - post_warm_time)
        if post_warm_time is not None else session_avg_speed
    )
    TRAINING_SUMMARY = {
        "reason": stop_reason, "step": step, "tokens_seen": tokens_seen,
        "target_tokens": CONFIG["target_tokens"], "latest_val_loss": latest_val_loss,
        "baseline_val_loss": baseline_val_loss, "best_val_loss": best_val_loss,
        "avg_speed": avg_speed, "session_avg_speed": session_avg_speed,
        "elapsed_seconds": elapsed,
        "latest_path": str(LATEST_PATH), "output_dir": str(OUTPUT_DIR),
    }
    print(f"Training cell finished: reason={stop_reason}; steps={step:,}; "
          f"tokens={tokens_seen:,}; average speed={avg_speed:,.0f} tok/s.")
    print("Final phone notification will be sent after the storage audit in the next cell.")

except Exception as exc:
    print(f"[FATAL] {type(exc).__name__}: {exc}")
    try:
        optimizer.zero_grad(set_to_none=True)
        # save_latest serializes committed_cursor directly; no stream reopen is needed here.
        save_latest("exception recovery")
    except Exception as save_exc:
        print(f"[checkpoint warning] {type(save_exc).__name__}: {save_exc}")
    notify("TinyGPT urgent error",
           f"Run: {RUN_NAME}\nError: {type(exc).__name__}: {exc}\n"
           f"Last safely committed step: {step:,}\nTokens: {tokens_seen:,}\n"
           f"Latest checkpoint: {LATEST_PATH}",
           priority="urgent", tags="rotating_light,warning")
    raise


Resuming: /kaggle/input/notebooks/fakesiddharth/pretrain-stage-4/checkpoints/stage4_latest.pt
Recovered prior best weights from attached Stage 4 output: /kaggle/input/notebooks/fakesiddharth/pretrain-stage-4/checkpoints/stage4_best.pt
Building deterministic validation cache (250,000 target tokens)...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (3683 > 1024). Running this sequence through the model will result in indexing errors


Saved validation cache: /kaggle/working/checkpoints/validation_cache.pt (250,112 positions)
Restored Stage 4 model, AdamW moments, AMP scaler, token counters, and Stage 4 stream cursor.
Rebuilding stream and skipping 1,891,099 shuffled rows...
Warmup: 150 steps | scheduled steps: 61,036
Required average speed for target within 8 hours: 69,444 tok/s, before overhead.
[ntfy] Sent: TinyGPT Stage 4 pretraining started
Resume self-check passed: first resumed chunk differs from the original.


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
[transformers] We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
You may ignore this warning if your `pad_token_id` (50256) is identical to the `bos_token_id` (50256), `eos_token_id` (50256), or the `sep_token_id` (None), and your input is not padded.


[train] step 56,450 | 1,849,753,600/2,000,000,000 (92.49%) | loss 3.4453 | lr 0.0001378289 | rolling 35,817 tok/s | post-warmup 35,595 tok/s | ETA 1h 10m | elapsed 0h 00m | GPU0 0.78 GiB allocated, 8.82 GiB peak reserved | GPU1 0.02 GiB allocated, 8.43 GiB peak reserved
[checkpoint] stage4_latest.pt: step 56,500, tokens 1,851,392,000 (periodic)
[train] step 56,500 | 1,851,392,000/2,000,000,000 (92.57%) | loss 3.7401 | lr 0.0001358987 | rolling 34,554 tok/s | post-warmup 35,236 tok/s | ETA 1h 10m | elapsed 0h 01m | GPU0 0.78 GiB allocated, 8.82 GiB peak reserved | GPU1 0.02 GiB allocated, 8.43 GiB peak reserved
[train] step 56,550 | 1,853,030,400/2,000,000,000 (92.65%) | loss 3.6172 | lr 0.0001339691 | rolling 34,127 tok/s | post-warmup 34,418 tok/s | ETA 1h 11m | elapsed 0h 02m | GPU0 0.78 GiB allocated, 8.82 GiB peak reserved | GPU1 0.02 GiB allocated, 8.43 GiB peak reserved
[train] step 56,600 | 1,854,668,800/2,000,000,000 (92.73%) | loss 3.8415 | lr 0.0001320404 | rolling 33,721 tok

In [4]:
# ----------------------------- STORAGE AUDIT + FINAL NOTIFICATION -----------
from pathlib import Path

audit_dir = Path("/kaggle/working/checkpoints")
print("Checkpoint/storage audit")
print("=" * 72)
files = sorted(p for p in audit_dir.rglob("*") if p.is_file()) if audit_dir.exists() else []
total_bytes = 0
for p in files:
    size = p.stat().st_size
    total_bytes += size
    print(f"{p.name:45s} {size / (1024**2):10.1f} MiB")
total_gib = total_bytes / (1024**3)
print("-" * 72)
print(f"Total files: {len(files)}")
print(f"Total size: {total_gib:.3f} GiB")
print(f"Storage budget: {CONFIG['storage_limit_gib']:.1f} GiB")
storage_ok = total_gib <= CONFIG["storage_limit_gib"]
print("Storage budget check:", "PASS" if storage_ok else "WARNING: over budget")

latest = audit_dir / f"{CONFIG['run_name']}_latest.pt"
final_weights = audit_dir / f"{CONFIG['run_name']}_final.pt"
print("Latest full checkpoint:", "present" if latest.exists() else "MISSING")
print("Final weights:", "present" if final_weights.exists() else "not present")

if "TRAINING_SUMMARY" in globals():
    s = TRAINING_SUMMARY
    if s["reason"] == "target_reached":
        title = "TinyGPT Stage 4 pretraining completed"
        message = (
            f"Run: {CONFIG['run_name']}\nReason: target reached\nSteps: {s['step']:,}\n"
            f"Tokens: {s['tokens_seen']:,}/{s['target_tokens']:,}\n"
            f"Latest validation loss: {s['latest_val_loss']}\n"
            f"Stage 4 starting baseline validation loss: {s['baseline_val_loss']}\n"
            f"Best validation loss: {s['best_val_loss']}\n"
            f"Post-warmup average speed (first 30 steps excluded): {s['avg_speed']:,.0f} tokens/sec\n"
            f"Overall session throughput: {s['session_avg_speed']:,.0f} tokens/sec\n"
            f"Session runtime: {fmt_duration(s['elapsed_seconds'])}\n"
            f"Storage audit: {'PASS' if storage_ok else 'OVER BUDGET'} ({total_gib:.3f} GiB)\n"
            f"Latest checkpoint: {latest}\nFinal weights: {final_weights}"
        )
        notify(title, message, priority="high", tags="tada,white_check_mark")
    else:
        title = "TinyGPT session ended - resume needed"
        message = (
            "session ended, resume needed: attach this output as input and rerun\n"
            f"Run: {CONFIG['run_name']}\nReason: {s['reason']}\nSteps: {s['step']:,}\n"
            f"Tokens: {s['tokens_seen']:,}/{s['target_tokens']:,}\n"
            f"Latest validation loss: {s['latest_val_loss']}\n"
            f"Stage 4 starting baseline validation loss: {s['baseline_val_loss']}\n"
            f"Post-warmup average speed (first 30 steps excluded): {s['avg_speed']:,.0f} tokens/sec\n"
            f"Overall session throughput: {s['session_avg_speed']:,.0f} tokens/sec\n"
            f"Session runtime: {fmt_duration(s['elapsed_seconds'])}\n"
            f"Storage audit: {'PASS' if storage_ok else 'OVER BUDGET'} ({total_gib:.3f} GiB)\n"
            f"Latest checkpoint: {latest}"
        )
        notify(title, message, priority="high", tags="warning,repeat")
else:
    print("TRAINING_SUMMARY unavailable; training may have failed before clean finalization.")
    print("The training cell should already have sent an urgent error notification if it caught an exception.")

print("\nFor a later Kaggle session, attach the saved output/checkpoint as an input dataset.")


Checkpoint/storage audit
stage4_best.pt                                     293.1 MiB
stage4_final.pt                                    293.1 MiB
stage4_latest.pt                                   682.9 MiB
stage4_milestone_2000M.pt                          293.1 MiB
validation_cache.pt                                  1.0 MiB
------------------------------------------------------------------------
Total files: 5
Total size: 1.526 GiB
Storage budget: 5.0 GiB
Storage budget check: PASS
Latest full checkpoint: present
Final weights: present
[ntfy] Sent: TinyGPT Stage 4 pretraining completed

For a later Kaggle session, attach the saved output/checkpoint as an input dataset.
